# AI Solution Architect Agent (Terminal-Demo)
Duenne Kommandozeilen-Demo. Die gesamte Logik lebt in **`architect.py`**
(Single Source of Truth) – dieselbe Datei, die auch `app.py` (Streamlit-UI)
nutzt. Aenderungen muessen daher nur noch in `architect.py` vorgenommen werden.

**Voraussetzung:** `./chroma_db` existiert (einmal via `Rag_Setup.ipynb` gebaut).

## 1) SETUP – Modell & Datenbank aus architect.py laden

In [ ]:
from architect import get_model, get_db, send_message

model = get_model()
conn = get_db()

count = conn.execute('SELECT COUNT(*) FROM conversations').fetchone()[0]
print("Setup abgeschlossen. Modell bereit.")
print(f"Gespeicherte Nachrichten in DB: {count}")

## 2) RAG-Kurztest – Wissensbasis abfragen
Prueft, ob die Chroma-Vektordatenbank erreichbar ist und Treffer liefert.

In [ ]:
from architect import search_patterns

print(search_patterns("Microservices")[:300] + "...")

## 3) CHAT-LOOP – Interaktives Gespraech mit dem Agent

Tippe deine Nachricht und druecke Enter. Mit `quit` beendest du den Chat.

In [ ]:
print("="*60)
print("AI Solution Architect – Chat gestartet")
print("Tippe 'quit' zum Beenden, 'history' fuer Chat-Verlauf")
print("="*60)

while True:
    user_input = input("\nDu: ").strip()
    
    if not user_input:
        continue
    
    if user_input.lower() == "quit":
        print("\nChat beendet. Auf Wiedersehen!")
        break
    
    if user_input.lower() == "history":
        print("\n--- Chat-Verlauf ---")
        for mid, role, content, ts in conn.execute(
            "SELECT id, role, content, timestamp FROM conversations ORDER BY id ASC"
        ).fetchall():
            label = "Du" if role == "user" else "Architect"
            print(f"[{ts[:19]}] {label}: {content[:100]}...")
        print("--- Ende ---")
        continue
    
    try:
        answer, in_tok, out_tok = send_message(conn, model, user_input)
        print(f"\n[Tokens: input={in_tok}, output={out_tok}]")
        print(f"\nArchitect: {answer}")
    except Exception as e:
        print(f"\nFehler: {e}")

conn.close()
print("Datenbankverbindung geschlossen.")